In [19]:
import duckdb
import polars as pl
import plotly.express as px
import numpy as np

conn = duckdb.connect("../data/mbta.duckdb", read_only=True)

def query(sql: str) -> pl.DataFrame:
    return conn.execute(sql).pl()

# Check current data volume
print(query("""
    SELECT 
        count(*) as total_predictions,
        count(DISTINCT date_trunc('hour', extracted_at)) as hours_collected,
        min(extracted_at) as first_extraction,
        max(extracted_at) as last_extraction
    FROM intermediate.int_scheduled_vs_actual
"""))

shape: (1, 4)
┌───────────────────┬─────────────────┬────────────────────────────┬────────────────────────────┐
│ total_predictions ┆ hours_collected ┆ first_extraction           ┆ last_extraction            │
│ ---               ┆ ---             ┆ ---                        ┆ ---                        │
│ i64               ┆ i64             ┆ datetime[μs]               ┆ datetime[μs]               │
╞═══════════════════╪═════════════════╪════════════════════════════╪════════════════════════════╡
│ 15644             ┆ 2               ┆ 2026-04-19 15:11:41.943332 ┆ 2026-04-19 16:25:49.263184 │
└───────────────────┴─────────────────┴────────────────────────────┴────────────────────────────┘


In [20]:
# Build the core feature table from intermediate layer
features = query("""
    WITH base AS (
        SELECT
            sva.prediction_id,
            sva.route_id,
            sva.route_name,
            sva.route_type,
            sva.route_type_desc,
            sva.stop_id,
            sva.stop_name,
            sva.municipality,
            sva.direction_id,
            sva.stop_sequence,
            sva.delay_seconds,
            sva.delay_category,
            sva.is_late,
            sva.is_significantly_late,
            sva.extracted_at,
            
            -- Temporal features
            extract(hour FROM sva.extracted_at) AS hour_of_day,
            extract(dow FROM sva.extracted_at) AS day_of_week,
            extract(minute FROM sva.extracted_at) AS minute_of_hour,
            
            CASE 
                WHEN extract(dow FROM sva.extracted_at) IN (0, 6) THEN 1 
                ELSE 0 
            END AS is_weekend,
            
            CASE
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 6 AND 9 THEN 'morning_rush'
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 10 AND 15 THEN 'midday'
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 16 AND 19 THEN 'evening_rush'
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 20 AND 23 THEN 'evening'
                ELSE 'overnight'
            END AS time_period,
            
            -- Stop position features
            CASE
                WHEN sva.stop_sequence <= 5 THEN 'early'
                WHEN sva.stop_sequence <= 15 THEN 'middle'
                ELSE 'late'
            END AS stop_position,
            
            -- Uncertainty features (from predictions)
            sva.arrival_uncertainty,
            sva.departure_uncertainty
            
        FROM intermediate.int_scheduled_vs_actual sva
        WHERE sva.delay_seconds IS NOT NULL
    )
    SELECT * FROM base
""")

print(f"Feature table shape: {features.shape}")
print(f"\nColumns: {features.columns}")
print(f"\nSample:")
print(features.head(5))

Feature table shape: (14393, 23)

Columns: ['prediction_id', 'route_id', 'route_name', 'route_type', 'route_type_desc', 'stop_id', 'stop_name', 'municipality', 'direction_id', 'stop_sequence', 'delay_seconds', 'delay_category', 'is_late', 'is_significantly_late', 'extracted_at', 'hour_of_day', 'day_of_week', 'minute_of_hour', 'is_weekend', 'time_period', 'stop_position', 'arrival_uncertainty', 'departure_uncertainty']

Sample:
shape: (5, 23)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ predictio ┆ route_id ┆ route_nam ┆ route_typ ┆ … ┆ time_peri ┆ stop_posi ┆ arrival_u ┆ departure │
│ n_id      ┆ ---      ┆ e         ┆ e         ┆   ┆ od        ┆ tion      ┆ ncertaint ┆ _uncertai │
│ ---       ┆ str      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ y         ┆ nty       │
│ str       ┆          ┆ str       ┆ i32       ┆   ┆ str       ┆ str       ┆ ---       ┆ ---       │
│           ┆          ┆           ┆           ┆ 

In [21]:
# Historical route performance features
route_features = query("""
    SELECT
        route_id,
        count(*) AS route_total_obs,
        round(avg(delay_seconds), 2) AS route_avg_delay,
        round(median(delay_seconds), 2) AS route_median_delay,
        round(stddev(delay_seconds), 2) AS route_stddev_delay,
        round(avg(CASE WHEN is_late THEN 1.0 ELSE 0.0 END), 4) AS route_late_rate,
        round(avg(CASE WHEN is_significantly_late THEN 1.0 ELSE 0.0 END), 4) AS route_sig_late_rate,
        round(percentile_cont(0.90) WITHIN GROUP (ORDER BY delay_seconds), 2) AS route_p90_delay
    FROM intermediate.int_scheduled_vs_actual
    WHERE delay_seconds IS NOT NULL
    GROUP BY route_id
""")

print("Route-Level Features:")
print(route_features.sort("route_avg_delay", descending=True))

Route-Level Features:
shape: (7, 8)
┌──────────┬────────────┬────────────┬────────────┬────────────┬───────────┬───────────┬───────────┐
│ route_id ┆ route_tota ┆ route_avg_ ┆ route_medi ┆ route_stdd ┆ route_lat ┆ route_sig ┆ route_p90 │
│ ---      ┆ l_obs      ┆ delay      ┆ an_delay   ┆ ev_delay   ┆ e_rate    ┆ _late_rat ┆ _delay    │
│ str      ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---       ┆ e         ┆ ---       │
│          ┆ i64        ┆ f64        ┆ f64        ┆ f64        ┆ f64       ┆ ---       ┆ f64       │
│          ┆            ┆            ┆            ┆            ┆           ┆ f64       ┆           │
╞══════════╪════════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ Blue     ┆ 982        ┆ 446.17     ┆ 518.0      ┆ 343.85     ┆ 0.8289    ┆ 0.667     ┆ 861.9     │
│ Green-D  ┆ 1230       ┆ 250.16     ┆ 239.0      ┆ 341.52     ┆ 0.778     ┆ 0.3317    ┆ 728.2     │
│ Green-C  ┆ 1415       ┆ 174.97     ┆ 140.0      ┆ 258

In [22]:
# Historical stop performance features
stop_features = query("""
    SELECT
        stop_id,
        count(*) AS stop_total_obs,
        round(avg(delay_seconds), 2) AS stop_avg_delay,
        round(median(delay_seconds), 2) AS stop_median_delay,
        round(stddev(delay_seconds), 2) AS stop_stddev_delay,
        round(avg(CASE WHEN is_late THEN 1.0 ELSE 0.0 END), 4) AS stop_late_rate
    FROM intermediate.int_scheduled_vs_actual
    WHERE delay_seconds IS NOT NULL
    GROUP BY stop_id
""")

print(f"Stop features: {stop_features.shape}")
print(stop_features.sort("stop_avg_delay", descending=True).head(10))

Stop features: (246, 6)
shape: (10, 6)
┌─────────┬────────────────┬────────────────┬──────────────────┬──────────────────┬────────────────┐
│ stop_id ┆ stop_total_obs ┆ stop_avg_delay ┆ stop_median_dela ┆ stop_stddev_dela ┆ stop_late_rate │
│ ---     ┆ ---            ┆ ---            ┆ y                ┆ y                ┆ ---            │
│ str     ┆ i64            ┆ f64            ┆ ---              ┆ ---              ┆ f64            │
│         ┆                ┆                ┆ f64              ┆ f64              ┆                │
╞═════════╪════════════════╪════════════════╪══════════════════╪══════════════════╪════════════════╡
│ 70056   ┆ 61             ┆ 575.1          ┆ 610.0            ┆ 381.59           ┆ 0.8197         │
│ 70058   ┆ 61             ┆ 563.61         ┆ 599.0            ┆ 381.74           ┆ 0.8197         │
│ 70060   ┆ 61             ┆ 548.52         ┆ 584.0            ┆ 381.99           ┆ 0.8197         │
│ 70052   ┆ 53             ┆ 512.94         ┆ 609.0 

In [23]:
# Join weather data
weather_features = query("""
    SELECT
        wt.prediction_id,
        wt.temperature_f,
        wt.humidity_pct,
        wt.precipitation_mm,
        wt.rain_mm,
        wt.snowfall_cm,
        wt.wind_speed_mph,
        wt.wind_gusts_mph,
        wt.visibility_m,
        wt.weather_code,
        wt.weather_condition,
        wt.is_precipitation,
        wt.is_snow,
        wt.is_low_visibility,
        wt.is_high_wind,
        wt.delay_seconds
    FROM intermediate.int_weather_transit wt
    WHERE wt.delay_seconds IS NOT NULL
""")

print(f"Weather-matched predictions: {weather_features.shape}")
print(f"\nWeather condition distribution:")
print(
    weather_features.group_by("weather_condition")
    .agg([
        pl.count().alias("count"),
        pl.col("delay_seconds").mean().round(1).alias("avg_delay"),
    ])
    .sort("avg_delay", descending=True)
)

Weather-matched predictions: (14393, 16)

Weather condition distribution:
shape: (3, 3)
┌───────────────────┬───────┬───────────┐
│ weather_condition ┆ count ┆ avg_delay │
│ ---               ┆ ---   ┆ ---       │
│ str               ┆ u32   ┆ f64       │
╞═══════════════════╪═══════╪═══════════╡
│ Rain              ┆ 12806 ┆ -21.1     │
│ Cloudy            ┆ 1581  ┆ -167.7    │
│ Drizzle           ┆ 6     ┆ -265.3    │
└───────────────────┴───────┴───────────┘


/var/folders/3w/csvlpk454tq67v7qllhl7ghm0000gr/T/ipykernel_71287/3900823466.py:29: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("count"),


In [24]:
alert_features = query("""
    WITH exploded AS (
        SELECT
            unnest(affected_routes) AS route_id,
            severity,
            informed_entity_count
        FROM marts.mart_alert_summary
        WHERE is_active = true
          AND affected_routes IS NOT NULL
    )
    SELECT
        route_id,
        count(*) AS active_alerts,
        max(severity) AS max_alert_severity,
        sum(informed_entity_count) AS total_entities_affected
    FROM exploded
    GROUP BY route_id
""")

print("Active Alert Features by Route:")
if len(alert_features) > 0:
    print(alert_features.sort("active_alerts", descending=True))
else:
    print("No active alerts with route data")

Active Alert Features by Route:


shape: (67, 4)
┌──────────┬───────────────┬────────────────────┬─────────────────────────┐
│ route_id ┆ active_alerts ┆ max_alert_severity ┆ total_entities_affected │
│ ---      ┆ ---           ┆ ---                ┆ ---                     │
│ str      ┆ i64           ┆ i32                ┆ decimal[38,0]           │
╞══════════╪═══════════════╪════════════════════╪═════════════════════════╡
│ Red      ┆ 8             ┆ 3                  ┆ 1805                    │
│ 429      ┆ 5             ┆ 7                  ┆ 125                     │
│ 435      ┆ 5             ┆ 7                  ┆ 125                     │
│ Orange   ┆ 5             ┆ 3                  ┆ 15                      │
│ 442      ┆ 4             ┆ 3                  ┆ 123                     │
│ …        ┆ …             ┆ …                  ┆ …                       │
│ 66       ┆ 1             ┆ 3                  ┆ 121                     │
│ 57       ┆ 1             ┆ 5                  ┆ 7                     

In [25]:
# Assemble the full feature matrix
feature_matrix = query("""
    WITH base AS (
        SELECT
            sva.prediction_id,
            sva.route_id,
            sva.stop_id,
            sva.direction_id,
            sva.stop_sequence,
            sva.delay_seconds,
            sva.is_late,
            
            -- Target
            sva.delay_seconds AS target_delay_seconds,
            CASE WHEN sva.delay_seconds > 60 THEN 1 ELSE 0 END AS target_is_late,
            
            -- Temporal
            extract(hour FROM sva.extracted_at) AS hour_of_day,
            extract(dow FROM sva.extracted_at) AS day_of_week,
            CASE WHEN extract(dow FROM sva.extracted_at) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,
            
            CASE
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 6 AND 9 THEN 1 ELSE 0
            END AS is_morning_rush,
            CASE
                WHEN extract(hour FROM sva.extracted_at) BETWEEN 16 AND 19 THEN 1 ELSE 0
            END AS is_evening_rush,
            
            -- Stop position
            sva.stop_sequence AS raw_stop_sequence,
            CASE
                WHEN sva.stop_sequence <= 5 THEN 0
                WHEN sva.stop_sequence <= 15 THEN 1
                ELSE 2
            END AS stop_position_bin,
            
            -- Uncertainty
            coalesce(sva.arrival_uncertainty, sva.departure_uncertainty, 0) AS uncertainty
            
        FROM intermediate.int_scheduled_vs_actual sva
        WHERE sva.delay_seconds IS NOT NULL
    ),
    
    route_agg AS (
        SELECT
            route_id,
            avg(delay_seconds) AS route_avg_delay,
            stddev(delay_seconds) AS route_stddev_delay,
            avg(CASE WHEN is_late THEN 1.0 ELSE 0.0 END) AS route_late_rate
        FROM intermediate.int_scheduled_vs_actual
        WHERE delay_seconds IS NOT NULL
        GROUP BY route_id
    ),
    
    stop_agg AS (
        SELECT
            stop_id,
            avg(delay_seconds) AS stop_avg_delay,
            stddev(delay_seconds) AS stop_stddev_delay,
            avg(CASE WHEN is_late THEN 1.0 ELSE 0.0 END) AS stop_late_rate
        FROM intermediate.int_scheduled_vs_actual
        WHERE delay_seconds IS NOT NULL
        GROUP BY stop_id
    )
    
    SELECT
        b.*,
        
        -- Route aggregate features
        coalesce(ra.route_avg_delay, 0) AS route_avg_delay,
        coalesce(ra.route_stddev_delay, 0) AS route_stddev_delay,
        coalesce(ra.route_late_rate, 0) AS route_late_rate,
        
        -- Stop aggregate features
        coalesce(sa.stop_avg_delay, 0) AS stop_avg_delay,
        coalesce(sa.stop_stddev_delay, 0) AS stop_stddev_delay,
        coalesce(sa.stop_late_rate, 0) AS stop_late_rate,
        
        -- Weather (join via int_weather_transit)
        coalesce(wt.temperature_f, 0) AS temperature_f,
        coalesce(wt.precipitation_mm, 0) AS precipitation_mm,
        coalesce(wt.wind_speed_mph, 0) AS wind_speed_mph,
        coalesce(wt.visibility_m, 24140) AS visibility_m,
        coalesce(wt.is_precipitation, false) AS is_precipitation,
        coalesce(wt.is_snow, false) AS is_snow
        
    FROM base b
    LEFT JOIN route_agg ra ON b.route_id = ra.route_id
    LEFT JOIN stop_agg sa ON b.stop_id = sa.stop_id
    LEFT JOIN intermediate.int_weather_transit wt ON b.prediction_id = wt.prediction_id
""")

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"\nFeature columns ({len(feature_matrix.columns)}):")
for col in feature_matrix.columns:
    dtype = feature_matrix[col].dtype
    nulls = feature_matrix[col].null_count()
    print(f"  {col:<30} {str(dtype):<15} nulls={nulls}")

Feature matrix shape: (69963, 29)

Feature columns (29):
  prediction_id                  String          nulls=0
  route_id                       String          nulls=0
  stop_id                        String          nulls=0
  direction_id                   Int32           nulls=0
  stop_sequence                  Int32           nulls=0
  delay_seconds                  Float64         nulls=0
  is_late                        Boolean         nulls=0
  target_delay_seconds           Float64         nulls=0
  target_is_late                 Int32           nulls=0
  hour_of_day                    Int64           nulls=0
  day_of_week                    Int64           nulls=0
  is_weekend                     Int32           nulls=0
  is_morning_rush                Int32           nulls=0
  is_evening_rush                Int32           nulls=0
  raw_stop_sequence              Int32           nulls=0
  stop_position_bin              Int32           nulls=0
  uncertainty                  

In [26]:
# Correlation with target
numeric_cols = [
    "hour_of_day", "day_of_week", "is_weekend",
    "is_morning_rush", "is_evening_rush",
    "raw_stop_sequence", "stop_position_bin", "uncertainty",
    "route_avg_delay", "route_stddev_delay", "route_late_rate",
    "stop_avg_delay", "stop_stddev_delay", "stop_late_rate",
    "temperature_f", "precipitation_mm", "wind_speed_mph", "visibility_m",
    "target_delay_seconds"
]

corr_df = feature_matrix.select(numeric_cols).to_pandas()
correlations = corr_df.corr()["target_delay_seconds"].drop("target_delay_seconds").sort_values(ascending=False)

print("Feature Correlations with Delay (seconds):")
print(correlations.to_string())

fig = px.bar(
    x=correlations.values,
    y=correlations.index,
    orientation="h",
    title="Feature Correlation with Delay (seconds)",
    labels={"x": "Pearson Correlation", "y": "Feature"},
)
fig.update_layout(height=600, yaxis={"categoryorder": "total ascending"})
fig.show()

Feature Correlations with Delay (seconds):
stop_avg_delay        0.774510
route_avg_delay       0.663639
stop_late_rate        0.619789
route_late_rate       0.608252
stop_stddev_delay     0.323301
route_stddev_delay    0.175019
hour_of_day           0.023644
is_evening_rush       0.023644
wind_speed_mph        0.019404
precipitation_mm     -0.026417
stop_position_bin    -0.048284
raw_stop_sequence    -0.115685
visibility_m         -0.147382
temperature_f        -0.155180
uncertainty          -0.218104
day_of_week                NaN
is_weekend                 NaN
is_morning_rush            NaN


In [27]:
# Compare distributions for late vs on-time
comparison_features = ["hour_of_day", "stop_avg_delay", "route_avg_delay", "uncertainty", "temperature_f", "wind_speed_mph"]

fm_pd = feature_matrix.select(comparison_features + ["target_is_late"]).to_pandas()
fm_pd["status"] = fm_pd["target_is_late"].map({0: "On Time", 1: "Late"})

for feat in comparison_features:
    fig = px.histogram(
        fm_pd,
        x=feat,
        color="status",
        barmode="overlay",
        title=f"{feat} Distribution: Late vs On-Time",
        opacity=0.7,
        color_discrete_map={"On Time": "#2ecc71", "Late": "#e74c3c"},
    )
    fig.show()

In [28]:
# Save for modeling notebook
feature_matrix.write_parquet("../data/feature_matrix.parquet")
print(f"Feature matrix saved: {feature_matrix.shape}")

print(f"""
FEATURE SUMMARY
{'='*50}
Total features: {len(feature_matrix.columns) - 3}  (excluding prediction_id and targets)
Total samples:  {len(feature_matrix)}
Late rate:      {feature_matrix['target_is_late'].mean():.1%}

Feature Groups:
  Temporal:     hour_of_day, day_of_week, is_weekend, rush hour flags
  Spatial:      stop_sequence, stop_position_bin
  Route hist:   route_avg_delay, route_stddev_delay, route_late_rate
  Stop hist:    stop_avg_delay, stop_stddev_delay, stop_late_rate
  Uncertainty:  arrival/departure uncertainty
  Weather:      temperature, precipitation, wind, visibility

NEXT: Notebook 04 — Train delay prediction models
""")

conn.close()

Feature matrix saved: (69963, 29)

FEATURE SUMMARY
Total features: 26  (excluding prediction_id and targets)
Total samples:  69963
Late rate:      25.4%

Feature Groups:
  Temporal:     hour_of_day, day_of_week, is_weekend, rush hour flags
  Spatial:      stop_sequence, stop_position_bin
  Route hist:   route_avg_delay, route_stddev_delay, route_late_rate
  Stop hist:    stop_avg_delay, stop_stddev_delay, stop_late_rate
  Uncertainty:  arrival/departure uncertainty
  Weather:      temperature, precipitation, wind, visibility

NEXT: Notebook 04 — Train delay prediction models

